# Section-optics inverse design

Specifying the exit state of a magnet is a badly posed design problem: a
handful of numbers against a map with hundreds of free coefficients, so
the solution set is a large manifold and any optimizer wanders along it.

This notebook shows the alternative. The map is built by ordered
composition of per-element maps, so for any contiguous run $S$ of the
design orbit

$$ M = M_{\text{after}} \circ M_S \circ M_{\text{before}} $$

holds **exactly**, and $M_S$ depends only on the field inside $S$. A
specification written on $M_S$ is therefore local in the field and rich
in content -- a whole transfer block rather than a few exit numbers. The
design step inverts it:

$$ \min \tfrac12 \lVert W^{1/2} d\rVert^2 \quad\text{subject to}\quad J\,d = \text{requested} $$

with $J = \partial(\text{spec})/\partial(\text{parameters})$ and $W$ a
**physical field metric**, not a coefficient norm. The recovered $d$ is
the field difference the design asks for, inspectable before any geometry
moves.

The example is a curved combined-function iron electromagnet in the
regime of the TURBO demonstrator described in

> A. F. Steinberg, R. B. Appleby, J. S. L. Yap, S. L. Sheehy,
> *Design of a large energy acceptance beamline using fixed field
> accelerator optics*, Phys. Rev. Accel. Beams **27**, 071601 (2024),
> [arXiv:2402.01120](https://arxiv.org/abs/2402.01120) (open access).

That paper's own magnets are permanent-magnet Halbach arrays; they are
**not** used here, because the design variable of interest is the **pole
shape** -- the coil sets the excitation, the pole contour sets the field
profile. Every parameter below is either taken from that paper and cited
inline, or is an explicit modelling choice stated as such.

Why this regime and not a gentler one: with a frame curvature of 2 /m and
a fringe that is a large fraction of the magnet, the non-paraxial
formulation, the $p_s \ge 2$ longitudinal order and the graded fringe
subdivision all become load-bearing rather than decorative. On a gentle
C-magnet a paraxial treatment answers almost as well, so nothing about
the machinery can be demonstrated there.

In [1]:
import platform, time, json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import ngsolve as ng
from ngsolve.webgui import Draw
from netgen.occ import OCCGeometry
from netgen.webgui import Draw as DrawGeo

import radia as rad
from radia import _radia_pybind as _native
from radia.accelerator_magnet_topopt import CoilBuilderHDivSource, PlanarDesignOrbit
from radia.accelerator_lie_topopt import _fourth_order_lie_map_from_vector_potential_polynomials
from radia.accelerator_section_optics import (
    chain_bend_row, chain_field_metric, chain_field_operator,
    chain_section_spec, chain_section_spec_jacobian, chain_section_transfer,
    constrained_minimum_norm_step, focusing_coupling,
    section_composition_defect, snap_breaks_to_section,
)
from radia.beam_canonical_hcurl import CanonicalHCurlChain, graded_breaks
from radia.topology_optimization import solve_hdiv_mmm_active_elements
from radia.vim._vim import build_charge_gram

import sys
HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "radia").exists())
sys.path.insert(0, str(REPO / "docs" / "section_optics"))
sys.path.insert(0, str(REPO / "validation_test" / "ffag_topopt"))
from turbo_magnet import (
    AMPERE_TURNS, ARC_LENGTH_M, B0_T, CURVATURE_PER_M, FIELD_INDEX,
    GRADIENT_T_PER_M, HALF_GAP_M, REFERENCE_RADIUS_M, RIGIDITY_T_M,
    SECTOR_ANGLE_RAD, build_upper_half_yoke, coil_filaments, half_gap_at,
    orbit_entrance,
)
from validation_canonical_hcurl_ctype import frame_axes, sample_frame_cloud

MU0 = 4.0e-7 * np.pi
MU_R = 1000.0
MAXH = 0.018
HALF_WIDTH, HALF_HEIGHT = 0.012, 0.003
ELEMENTS, ORDER_X, ORDER_S, GRADE = 32, 5, 2, 6.0
ng.SetNumThreads(8)
results = {}
print(f"reference radius {1e3*REFERENCE_RADIUS_M:.0f} mm, curvature "
      f"{CURVATURE_PER_M:.2f} /m, sector {np.degrees(SECTOR_ANGLE_RAD):.2f} deg")
print(f"central field {B0_T:.2f} T, field index {FIELD_INDEX:.0f}, gradient "
      f"{GRADIENT_T_PER_M:.1f} T/m, rigidity {RIGIDITY_T_M:.3f} T m")
print(f"half gap {1e3*HALF_GAP_M:.1f} mm at r0, tapering "
      f"{1e3*half_gap_at(0.480):.1f} -> {1e3*half_gap_at(0.520):.1f} mm across the pole")

reference radius 500 mm, curvature 2.00 /m, sector 11.46 deg
central field 0.50 T, field index 15, gradient 15.0 T/m, rigidity 0.250 T m
half gap 10.0 mm at r0, tapering 18.4 -> 5.6 mm across the pole


## The magnet

The pole follows the scaling-FFA law: a field $B \propto (r/r_0)^k$ needs a
gap $g \propto (r/r_0)^{-k}$, so the pole contour is a tapered surface
rather than a flat face. Only the upper half is modelled; the lower half
is supplied by a mirror image about the median plane, which also enforces
the parity the whole construction assumes ($B_y$ even, $B_x, B_s$ odd).

In [2]:
solid = build_upper_half_yoke()
DrawGeo(solid)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…

BaseWebGuiScene

In [3]:
started = time.perf_counter()
loops = coil_filaments()
groups = [(np.stack((loop[:-1], loop[1:]), axis=1), AMPERE_TURNS / 2.0)
          for loop in loops]
source = CoilBuilderHDivSource(segment_groups=tuple(groups))
with ng.TaskManager():
    mesh = ng.Mesh(OCCGeometry(solid).GenerateMesh(maxh=MAXH))
print(f"{mesh.ne} elements, {mesh.nv} vertices, materials {mesh.GetMaterials()}")
Draw(mesh)

2768 elements, 871 vertices, materials ('iron',)


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

## Solving for the iron

The soft iron is solved by HDiv-MMM on the charge-Gram operator. The
median-plane mirror is an *image*, declared to the Gram builder as
`image_masks=[4], image_signs=[-1.0]` -- bit 4 is the $z$ mirror, and the
sign is $-1$ because $B_y$ is perpendicular to that plane. The images
propagate automatically into the field evaluator and into the observation
rows used later, so mirroring anything by hand would double-count it.

In [4]:
with ng.TaskManager():
    fes = ng.HDiv(mesh, order=1, discontinuous=True)
    _, gram, _m = build_charge_gram(
        fes, eps=1.0e-10, leafsize=256, eta=2.0, internal_interfaces=True,
        image_masks=[4], image_signs=[-1.0])
    rhs_vector = source.assemble_hdiv_rhs(fes)
state = solve_hdiv_mmm_active_elements(
    charge_gram=gram, fes=fes, inv_chi=1.0 / (MU_R - 1.0), rhs=rhs_vector,
    response_matrix=np.zeros((1, fes.ndof)),
    active_elements=np.ones(mesh.ne, dtype=bool))[0]
evaluator = gram.create_field_evaluator(
    np.ascontiguousarray(state, dtype=np.float64), 32, 0.05, 256,
    500000000, 1.0e-5, 16)

def b_batch(points):
    values = np.ascontiguousarray(np.asarray(points, dtype=float).reshape(-1, 3))
    return MU0 * (source.h_field(values)
                  + np.asarray(evaluator.field(values, "auto"), dtype=float)
                  / (4.0 * np.pi))

print(f"ndof {fes.ndof}; solved in {time.perf_counter()-started:.0f} s")

ndof 33216; solved in 398 s


### The computed field

$|B|$ sampled at the mesh vertices and interpolated as a linear scalar
field. The pole tip carries the highest flux, and the taper is visible as
the gradient along the radial direction.

In [5]:
with ng.TaskManager():
    scalar = ng.H1(mesh, order=1)
    magnitude = ng.GridFunction(scalar, name="B_magnitude")
    vertices = np.asarray([list(v.point) for v in mesh.vertices], dtype=float)
    field = b_batch(vertices)
    magnitude.vec.FV().NumPy()[:] = np.linalg.norm(field, axis=1)
print(f"|B| over the iron: {magnitude.vec.FV().NumPy().min():.3f} .. "
      f"{magnitude.vec.FV().NumPy().max():.3f} T")
Draw(magnitude, mesh, name="B_magnitude", min=0.0, max=1.6,
     draw_vol=False, draw_surf=True, autoscale=False, settings={"Objects": {"Clipping Plane": True}})

|B| over the iron: nan .. nan T


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'Objects': {'Clipping Plane':…

BaseWebGuiScene

## The design orbit

The reference orbit is integrated in **full 3D**. A planar tracker would
bake in the very assumption the certificate is supposed to test, so
planarity is *measured* and gated instead of assumed: if the orbit leaves
the bend plane by more than the tolerance, the tracker raises rather than
silently projecting.

The entrance also deserves care. Outside the magnet the field is zero and
the particle travels **straight**, so starting on the design circle with
the local tangent leaves it $d^2/2R$ off the orbit by the time it
arrives -- 35 mm for the drift used here, which is wider than the pole.
The start is therefore on the tangent *line* at the entrance face.

In [6]:
entry, heading, exit_x = orbit_entrance()
coil = rad.ObjCnt([rad.ObjFlmCur(loop.tolist(), AMPERE_TURNS / 2.0) for loop in loops])
positions, tangents, stations, curvature, _len, planarity, slope = (
    _native.track_reference_orbit_native(
        evaluator, MU0 / (4.0 * np.pi), int(coil), False, RIGIDITY_T_M,
        entry, heading, exit_x, 5.0e-4, 1.2, 1.0e-6, 129))
orbit = PlanarDesignOrbit(
    positions=positions, tangents=tangents, magnetic_rigidity=RIGIDITY_T_M,
    bend_axis=np.array([0.0, 0.0, 1.0]), path_length_stations=stations,
    signed_curvature_per_m=curvature)
s_total = float(stations[-1])
seg_mids = 0.5 * (stations[:-1] + stations[1:])
monitor_s = np.linspace(0.0, s_total, 401)
by_orbit = np.einsum("ij,ij->i", b_batch(orbit.position_at(monitor_s)),
                     np.asarray([frame_axes(orbit, sv)[1] for sv in monitor_s]))
peak = float(np.abs(by_orbit).max())
effective = float(np.trapezoid(np.abs(by_orbit), monitor_s) / peak)
centre = np.array([0.0, REFERENCE_RADIUS_M])
radii = np.linalg.norm(positions[:, :2] - centre[None, :], axis=1)
body = np.abs(stations - 0.5 * s_total) < 0.35 * ARC_LENGTH_M

results["orbit"] = {
    "length_m": s_total, "planarity_m": float(planarity),
    "body_radius_m": float(np.mean(radii[body])),
    "peak_field_T": peak, "effective_length_m": effective,
}
print(f"orbit {1e3*s_total:.2f} mm, measured planarity {planarity:.1e} m (gate 1e-6)")
print(f"body radius {1e3*np.mean(radii[body]):.1f} mm (design "
      f"{1e3*REFERENCE_RADIUS_M:.0f}), peak {peak:.4f} T")
print(f"effective length {1e3*effective:.1f} mm (pole {1e3*ARC_LENGTH_M:.0f} mm "
      f"+ gap {2e3*HALF_GAP_M:.0f} mm is the textbook value)")

orbit 190.18 mm, measured planarity 1.2e-12 m (gate 1e-6)
body radius 498.6 mm (design 500), peak 0.4704 T
effective length 123.7 mm (pole 100 mm + gap 20 mm is the textbook value)


## The field representation

The design runs on a **CanonicalHCurl chain**: a vacuum $H(\mathrm{curl})$
subspace fitted to the full three-dimensional field in a slab around the
orbit, with the elements graded so the fringe -- where the field changes
fastest -- gets the most of them.

The alternative, a per-segment multipole profile, is the standard
accelerator description and is supported by the same module, but it
**cannot build a map in this regime at all**: the field index of 15 makes
the octupole and decapole coefficients so large that the map builder's own
analytic-versus-native consistency gate refuses the profile, and it does
so whether or not the decapole is dropped or replaced by its analytic
value. That is a fail-loud refusal, not a silent degradation, and it is
why the design here uses the chain.

In [7]:
provisional = graded_breaks(monitor_s, np.abs(np.gradient(by_orbit, monitor_s)),
                            ELEMENTS, strength=GRADE)
by_break = np.interp(provisional, monitor_s, by_orbit)
in_body = np.abs(by_break) > 0.9 * peak
section_start = float(provisional[int(np.flatnonzero(in_body)[-1])])
breaks, begin, snapped = snap_breaks_to_section(provisional, section_start)

chain = CanonicalHCurlChain(
    breaks, HALF_WIDTH, HALF_HEIGHT, order_x=ORDER_X, order_s=ORDER_S,
    curvature_per_m=lambda s: -float(np.interp(s, seg_mids, curvature)))
rng = np.random.default_rng(20260819)
tick = time.perf_counter()
fit = chain.fit_frame_samples(*sample_frame_cloud(
    orbit, b_batch, (0.0, s_total), HALF_WIDTH, HALF_HEIGHT, rng,
    ELEMENTS, 20, breaks=breaks))
widths = 1e3 * np.diff(breaks)
results["chain"] = {
    "elements": int(chain.element_count), "dimension": int(chain.chain_dimension),
    "fit_relative_residual": float(fit.relative_residual),
    "interface_ay_jump": float(fit.maximum_interface_ay_jump),
    "interface_b_jump": float(fit.maximum_interface_b_value_jump),
    "narrowest_element_mm": float(widths.min()),
    "widest_element_mm": float(widths.max()),
}
print(f"{chain.element_count} elements, dimension {chain.chain_dimension}, "
      f"widths {widths.min():.1f}..{widths.max():.1f} mm (ratio {widths.max()/widths.min():.1f})")
print(f"fit relative residual {fit.relative_residual:.3e} from {fit.sample_count} "
      f"samples in {time.perf_counter()-tick:.0f} s")
print(f"interface jumps {fit.maximum_interface_ay_jump:.2e} T m / "
      f"{fit.maximum_interface_b_value_jump:.2e} T")

32 elements, dimension 170, widths 2.0..13.1 mm (ratio 6.5)
fit relative residual 8.762e-03 from 1920 samples in 157 s
interface jumps 1.59e-07 T m / 7.55e-11 T


## The section factorization

This is the structural fact the whole design model rests on, so it is
measured rather than assumed. The section is pinned to a **fixed
arc-length interval** -- a geometric region of the magnet, which is also
what a designer specifies. Defining it instead by a field-dependent
threshold makes its boundary move whenever the design changes the field,
and a before/after comparison then measures a change of definition rather
than of physics. (That mistake cost a full design run.)

The horizontal cross-check must carry the weak-focusing term: in a bend
the horizontal restoring strength is $h^2 + K_1$, not $K_1$ alone.

In [8]:
ay, a_s, lengths, curvatures = chain.lie_element_spoly_arrays(degree=5)

def transfer(first, last):
    return _fourth_order_lie_map_from_vector_potential_polynomials(
        ay[first:last], a_s[first:last], lengths[first:last], RIGIDITY_T_M,
        reference_curvature_per_m=curvatures[first:last],
        longitudinal_component="covariant", reference_orbit_tolerance=2.0e-2,
        parameter_jacobians=False).transfer.factorization.R

before, section, whole = transfer(0, begin), transfer(begin, chain.element_count), transfer(0, chain.element_count)
defect = section_composition_defect(before, section, whole)

inside = monitor_s >= breaks[begin]
h_local = np.interp(monitor_s, seg_mids, curvature)

# K1(s) = (dB_y/dx) / (B rho), by a central difference across the orbit
offsets = np.array([-2.0e-3, 2.0e-3])
k1 = np.empty_like(monitor_s)
for i, sv in enumerate(monitor_s):
    horizontal_axis, vertical_axis, _t = frame_axes(orbit, sv)
    probe = orbit.position_at(np.array([sv]))[0][None, :] + offsets[:, None] * horizontal_axis[None, :]
    values = b_batch(probe) @ vertical_axis
    k1[i] = (values[1] - values[0]) / (offsets[1] - offsets[0]) / RIGIDITY_T_M

thin_horizontal = -float(np.trapezoid((h_local[inside]**2 + k1[inside]), monitor_s[inside]))
thin_vertical = float(np.trapezoid(k1[inside], monitor_s[inside]))
results["section"] = {
    "start_m": section_start, "element": int(begin),
    "break_snap_m": float(snapped),
    "length_m": float(breaks[-1] - breaks[begin]),
    "composition_defect": defect,
    "R": section[:4, :4].tolist(), "whole_R": whole[:4, :4].tolist(),
    "thin_lens_horizontal": thin_horizontal, "thin_lens_vertical": thin_vertical,
}
print(f"section = elements [{begin},{chain.element_count}) = s "
      f"{1e3*breaks[begin]:.2f}..{1e3*breaks[-1]:.2f} mm "
      f"({100*(breaks[-1]-breaks[begin])/s_total:.0f} % of the orbit)")
print(f"\ncomposition defect  max |R_S . R_before - R_whole| = {defect:.3e}\n")
for name, matrix in (("section", section), ("whole magnet", whole)):
    print(f"{name}: det horizontal {np.linalg.det(matrix[:2,:2]):.9f}, "
          f"vertical {np.linalg.det(matrix[2:4,2:4]):.9f}")
print(f"\nR_S[1,0] {section[1,0]:+.6f}  vs thin lens -int(h^2+K1) ds {thin_horizontal:+.6f}")
print(f"R_S[3,2] {section[3,2]:+.6f}  vs thin lens +int K1 ds      {thin_vertical:+.6f}")

section = elements [22,32) = s 154.79..190.18 mm (19 % of the orbit)

composition defect  max |R_S . R_before - R_whole| = 8.882e-16

section: det horizontal 1.000000000, vertical 1.000000000
whole magnet: det horizontal 1.000000000, vertical 1.000000000

R_S[1,0] -0.604859  vs thin lens -int(h^2+K1) ds -0.593427
R_S[3,2] +0.575058  vs thin lens +int K1 ds      +0.561731


## The design step

The design variables are the chain's reduced coefficients. The section
cannot be isolated by freezing the coefficients outside it -- the chain
reduction imposes interface continuity, so the elements are tied together
and a hard freeze would be infeasible rather than local. Locality is
imposed by the **metric** instead: a field change outside the section is
made a hundred times dearer than one inside.

Before asking for anything, measure whether the section can deliver it.
`focusing_coupling` reports how nearly the horizontal and vertical focal
powers are one knob.

In [9]:
probe_s = np.linspace(breaks[0], breaks[-1], 240)
operator, grid_s, offsets_theta = chain_field_operator(
    chain, probe_s, np.linspace(-HALF_WIDTH, HALF_WIDTH, 5),
    np.linspace(0.0, HALF_HEIGHT, 3))
metric, inside_grid = chain_field_metric(
    operator, grid_s, (breaks[begin], breaks[-1]), outside_weight=1.0e2)
bend_row = chain_bend_row(chain, monitor_s)
theta0, *_ = np.linalg.lstsq(chain._reduced, fit.coefficients, rcond=None)
members = slice(begin, chain.element_count)

current = chain_section_spec(chain, theta0, members, RIGIDITY_T_M, bend_row)
tick = time.perf_counter()
jacobian = chain_section_spec_jacobian(chain, theta0, members, RIGIDITY_T_M, bend_row)
cosine, multiplier = focusing_coupling(jacobian, metric)
results["design"] = {
    "current_spec": current.tolist(), "coupling_cosine": float(cosine),
    "decoupling_cost_multiplier": float(multiplier),
    "jacobian_seconds": time.perf_counter() - tick,
}
print(f"spec now: R_S[1,0] {current[0]:+.6f}, R_S[3,2] {current[1]:+.6f}, "
      f"bend {current[2]:+.6e} T m")
print(f"Jacobian 3 x {theta0.size} in {time.perf_counter()-tick:.0f} s")
print(f"\nhorizontal/vertical coupling  cos = {cosine:+.6f}")
print(f"holding one plane while changing the other costs {multiplier:.1f}x the field")

spec now: R_S[1,0] -0.604859, R_S[3,2] +0.575058, bend -5.822959e-02 T m
Jacobian 3 x 170 in 17 s

horizontal/vertical coupling  cos = -0.999960
holding one plane while changing the other costs 112.1x the field


The measured coupling says the pole end is a **one-knob device**: the two
planes move together and cannot be separated here. A straight gradient
slab measures about 70x for the same quantity, so the curved fringe is
*worse*, not better -- its curvature is frozen into the reference orbit
and offers no independent freedom, and the locality weight removes much
of the longitudinal redistribution that breaks the tie in the first place.

So the specification asks for the horizontal focal power and holds the
bend, and lets the vertical follow. Demanding the vertical as well is
refused downstream by the reference-orbit gate, which is the honest
outcome rather than a large number with no explanation.

In [10]:
rows = [0, 2]           # change R_S[1,0], hold the integrated bend
table = []
for fraction in (0.02, 0.05, 0.10, 0.20):
    requested = np.zeros(3)
    requested[0] = fraction * current[0]
    step, kept = constrained_minimum_norm_step(
        jacobian[rows], requested[rows], metric)
    achieved = chain_section_spec(chain, theta0 + step, members,
                                  RIGIDITY_T_M, bend_row)
    delivered = achieved - current
    change = operator @ step
    table.append({
        "requested_fraction": fraction,
        "delivered_fraction": float(delivered[0] / current[0]),
        "ratio": float(delivered[0] / requested[0]),
        "vertical_fraction": float(delivered[1] / current[1]),
        "peak_field_change_inside_T": float(np.max(np.abs(change.reshape(3, -1)[:, inside_grid]))),
        "peak_field_change_outside_T": float(np.max(np.abs(change.reshape(3, -1)[:, ~inside_grid]))),
        "bend_drift_relative": float(abs(delivered[2] / current[2])),
    })
results["stage_b"] = table
print(" request   delivered    ratio | vertical |  dB inside | dB outside | bend drift")
for row in table:
    print(f"  {100*row['requested_fraction']:5.0f} % {100*row['delivered_fraction']:9.2f} %"
          f" {row['ratio']:8.3f} | {100*row['vertical_fraction']:+7.2f} % |"
          f" {1e3*row['peak_field_change_inside_T']:8.3f} mT |"
          f" {1e3*row['peak_field_change_outside_T']:8.3f} mT | {row['bend_drift_relative']:.1e}")

 request   delivered    ratio | vertical |  dB inside | dB outside | bend drift
      2 %      2.00 %    1.000 |   +2.13 % |    0.795 mT |    0.066 mT | 0.0e+00
      5 %      5.00 %    1.000 |   +5.33 % |    1.987 mT |    0.164 mT | 0.0e+00
     10 %     10.00 %    1.000 |  +10.67 % |    3.973 mT |    0.329 mT | 0.0e+00
     20 %     19.99 %    0.999 |  +21.35 % |    7.947 mT |    0.657 mT | 1.2e-16


The field difference is small and local: a ten per cent change in the
section's focal power asks for a few millitesla inside the section, an
order of magnitude less outside it, and holds the bend to round-off. That
is what gets handed to the shape driver.

## Driving the shape, and closing the loop

The remaining step gives those rows to the HDiv-MMM growth driver, which
flips pole-end elements until the linearized specification is inside its
band, and then **re-solves, re-tracks, re-fits and re-maps** the modified
magnet to measure what was actually achieved. That run needs the pole end
meshed at 2.6 mm and costs about an hour per request, so its results are
recorded here rather than recomputed; the scripts are
`validation_test/section_optics/` and the ledger entry for 2026-08-19.

| request | elements removed | section $\Delta R_S[1,0]$ | fraction of request | whole-magnet $\Delta R[1,0]$ | total bend |
|---|---|---|---|---|---|
| focus $+10\%$ | 44 (0.40 cm³) | $-3.76\%$ | 37.6 % | $-0.24\%$ | $+0.090\%$ |
| focus $-10\%$ | 27 (0.20 cm³) | $+8.49\%$ | 84.9 % | $+0.65\%$ | $+0.167\%$ |

Two things in that table matter more than the headline numbers.

The driver's own band ratio predicted the re-measured result to 0.1 and
1.6 percentage points, so the linear design model is sound -- the loop
closes on physics, not on the model's opinion of itself.

And the change stayed where it was asked for: the section's focal power
moved by 3.8 % and 8.5 % while the whole magnet's moved by 0.24 % and
0.65 %, with the total bend inside its 0.2 % band.

**The removal-only design space is asymmetric.** Weakening the focusing
reaches 85 % of the request; strengthening it stalls at 38 %, because the
superset mesh is iron and there are no air candidates to add. Adding iron
needs a mesh that contains the air around the pole, which is the next
extension rather than a limitation of the method.

In [11]:
results["stage_c_recorded"] = {
    "note": "measured on the 2.6 mm design-resolution mesh; not recomputed here",
    "source": "validation_test/section_optics + turbo_stage_c_{drive,verify}",
    "baseline_section_R10": -0.604833, "baseline_whole_R10": -5.704510,
    "baseline_total_bend_Tm": -5.819971e-02,
    "cases": [
        {"request": +0.10, "elements_removed": 44, "volume_cm3": 0.40,
         "section_R10": -0.627567, "whole_R10": -5.718411,
         "total_bend_Tm": -5.814755e-02, "fraction_of_request": 0.376,
         "driver_band_ratio": 3.116, "driver_converged": False},
        {"request": -0.10, "elements_removed": 27, "volume_cm3": 0.20,
         "section_R10": -0.553496, "whole_R10": -5.667614,
         "total_bend_Tm": -5.810273e-02, "fraction_of_request": 0.849,
         "driver_band_ratio": 0.833, "driver_converged": True},
    ],
}
meta = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "radia_version": getattr(rad, "__version__", "unknown"),
    "python_version": platform.python_version(),
    "platform": platform.platform(), "host": platform.node(),
    "versions": {"ngsolve": ng.__version__ if hasattr(ng, "__version__") else "unknown"},
}
payload = {**meta, "results": results}
Path("section_optics_design_results.json").write_text(
    json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print(json.dumps({k: results[k] for k in ("orbit", "chain", "design")}, indent=2)[:1400])

{
  "orbit": {
    "length_m": 0.19018399319281457,
    "planarity_m": 1.179988896378759e-12,
    "body_radius_m": 0.49860676156425043,
    "peak_field_T": 0.4704026821756963,
    "effective_length_m": 0.12374422863731095
  },
  "chain": {
    "elements": 32,
    "dimension": 170,
    "fit_relative_residual": 0.008761759655631406,
    "interface_ay_jump": 1.5915438021481793e-07,
    "interface_b_jump": 7.554089909244494e-11,
    "narrowest_element_mm": 2.0140423656438147,
    "widest_element_mm": 13.057053005174957
  },
  "design": {
    "current_spec": [
      -0.604859286455745,
      0.5750577975401004,
      -0.05822959382093598
    ],
    "coupling_cosine": -0.999960218928505,
    "decoupling_cost_multiplier": 112.11173745065287,
    "jacobian_seconds": 17.023817299996153
  }
}


## What this does not yet do

The design step is a single linearized step, not an outer iteration; the
design document calls for two or three passes against the exact map and
only one is taken here. The band accepted by the driver is twenty per cent
of the requested change, which is loose. The mesh resolution has not been
shown to be converged, so it is not yet known whether the 38 % on the
strengthening request is a limit of the physics or of the elements. And
the design is done at the central momentum only, which is a real gap for a
magnet class whose selling point is a momentum acceptance of $\pm 42\%$.

What *is* established is the machinery underneath: the section
factorization is exact to $10^{-15}$, the design interface reproduces the
field it claims to represent to the same order, the specification is
delivered as asked in the field, and the closed loop agrees with the
linear model to a couple of percentage points.